# Phase 4 Evaluation Notebook (Kaggle)

This notebook:
1. Installs dependencies
2. Copies dataset/script/model folder or zip from /kaggle/input to /kaggle/working
3. Prepares model artifacts
4. Runs evaluate_baselines_kaggle.py
5. Verifies outputs
6. Creates a zip for download

In [ ]:
# Cell 1: Install Dependencies
!pip -q install --upgrade pip
!pip -q install transformers torch scikit-learn tqdm numpy matplotlib seaborn safetensors sentencepiece

print('Dependencies installed.')

In [ ]:
# Cell 2: Copy Files to /kaggle/working
import shutil
import zipfile
from pathlib import Path

input_root = Path('/kaggle/input')
working_root = Path('/kaggle/working')

dataset_name = 'cleaned_augmented_neuralchemy_dataset.jsonl'
script_name = 'evaluate_baselines_kaggle.py'
model_zip_name = 'saved_model_artifacts.zip'
model_folder_name = 'saved_model_artifacts'

dataset_candidates = list(input_root.rglob(dataset_name))
script_candidates = list(input_root.rglob(script_name))
zip_candidates = list(input_root.rglob(model_zip_name))
model_folder_candidates = [p for p in input_root.rglob(model_folder_name) if p.is_dir()]

if not dataset_candidates:
    raise FileNotFoundError(f'Could not find {dataset_name} under /kaggle/input')
if not script_candidates:
    raise FileNotFoundError(f'Could not find {script_name} under /kaggle/input')

# Show discovered candidates for easier debugging
print('Dataset candidates:', [str(p) for p in dataset_candidates[:3]])
print('Script candidates :', [str(p) for p in script_candidates[:3]])
print('Zip candidates    :', [str(p) for p in zip_candidates[:3]])
print('Folder candidates :', [str(p) for p in model_folder_candidates[:3]])

dataset_src = dataset_candidates[0]
script_src = script_candidates[0]

dataset_dst = working_root / dataset_name
script_dst = working_root / script_name

shutil.copy2(dataset_src, dataset_dst)
shutil.copy2(script_src, script_dst)

print('Copied dataset :', dataset_dst)
print('Copied script  :', script_dst)

extract_dir = working_root / 'saved_model_extracted'
extract_dir.mkdir(parents=True, exist_ok=True)

# Preferred: Kaggle dataset auto-extracted folder already exists
if model_folder_candidates:
    model_folder_src = model_folder_candidates[0]
    model_folder_dst = extract_dir / model_folder_name
    if model_folder_dst.exists():
        shutil.rmtree(model_folder_dst)
    shutil.copytree(model_folder_src, model_folder_dst)
    print('Copied model folder:', model_folder_src)
    print('To                :', model_folder_dst)
# Fallback: use zip if present
elif zip_candidates:
    zip_src = zip_candidates[0]
    zip_dst = working_root / model_zip_name
    shutil.copy2(zip_src, zip_dst)
    print('Copied zip     :', zip_dst)
    with zipfile.ZipFile(zip_dst, 'r') as zf:
        zf.extractall(extract_dir)
    print('Extracted zip to:', extract_dir)
else:
    raise FileNotFoundError(
        f'Could not find either folder "{model_folder_name}" or zip "{model_zip_name}" under /kaggle/input'
    )

candidate_model_dirs = list(extract_dir.rglob('best_binary_injection'))
if not candidate_model_dirs:
    raise FileNotFoundError('Could not find best_binary_injection after copy/extract')

model_dir = candidate_model_dirs[0]
print('Model directory:', model_dir)

required = ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
missing = [f for f in required if not (model_dir / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing required model files in {model_dir}: {missing}')

print('Model files check passed.')

In [ ]:
# Cell 3: Verify GPU
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU count     :', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))

In [ ]:
# Cell 4: Run Evaluation Script (with real-time logging)
import subprocess
import os
from pathlib import Path

os.chdir('/kaggle/working')
output_dir = '/kaggle/working/phase4_results'

extract_dir = Path('/kaggle/working/saved_model_extracted')
model_dir = list(extract_dir.rglob('best_binary_injection'))[0]

cmd = [
    'python', '/kaggle/working/evaluate_baselines_kaggle.py',
    '--data', '/kaggle/working/cleaned_augmented_neuralchemy_dataset.jsonl',
    '--model', str(model_dir),
    '--output_dir', output_dir,
    '--epochs', '3',
    '--batch', '16',
    '--maxlen', '256'
]

print('Running command:')
print(' '.join(cmd))
print('\n' + '='*70)
print('Evaluation Output (Real-time Streaming):')
print('='*70 + '\n')

# Stream output in real-time instead of capturing it all
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, 
                        text=True, bufsize=1, universal_newlines=True)

output_text = []
for line in proc.stdout:
    print(line, end='')
    output_text.append(line)

returncode = proc.wait()

if returncode != 0:
    print('\n' + '='*70)
    print(f'ERROR: Script failed with exit code {returncode}')
    print('='*70)
    raise RuntimeError(f'evaluate_baselines_kaggle.py failed with exit code {returncode}')

print('\n' + '='*70)
print('Evaluation completed successfully!')
print('='*70)

In [ ]:
# Cell 5: Check Results
import json
from pathlib import Path

out = Path('/kaggle/working/phase4_results')
results_json = out / 'results_phase4.json'

if not out.exists():
    raise FileNotFoundError(f'Output directory not found: {out}')

print('Output files:')
for f in sorted(out.glob('*')):
    print('-', f.name)

if not results_json.exists():
    raise FileNotFoundError(f'Missing results file: {results_json}')

with open(results_json, 'r', encoding='utf-8') as f:
    results = json.load(f)

print('\nQuick summary:')
if 'full_model' in results:
    fm = results['full_model']
    print('Full model F1:', fm.get('f1'))
    print('Full model AUC:', fm.get('auc'))
if 'baseline_tfidf_lr' in results:
    b = results['baseline_tfidf_lr']
    print('TF-IDF baseline F1:', b.get('f1'))
if 'latency' in results:
    print('Latency mean (ms):', results['latency'].get('mean_ms'))

print('\nresults_phase4.json loaded successfully.')

In [ ]:
# Cell 6: Download Results
import shutil
from pathlib import Path

out_dir = Path('/kaggle/working/phase4_results')
zip_base = '/kaggle/working/phase4_results_archive'
zip_file = zip_base + '.zip'

if not out_dir.exists():
    raise FileNotFoundError(f'Cannot zip missing directory: {out_dir}')

shutil.make_archive(zip_base, 'zip', str(out_dir))
print('Created:', zip_file)
print('Download this file from the Kaggle notebook output panel.')